In [ ]:
!pip install git+https://github.com/openai/whisper.git #for speech to text transcription
!sudo apt update && sudo apt install ffmpeg # ffmeg - to process audio video files and convert and extract audio from audio and video files

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-vjecpmwg
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-vjecpmwg
  Resolved https://github.com/openai/whisper.git to commit c0d2f624c09dc18e709e37c2ad90c039a4eb72a2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=37cbd371f768dd196bcef745ee774875d4dacc019f9afe9998efc5739eafacac
  Stored in directory: /tmp/pip-ephem-wheel-cache-dtehmp_p/wheels/c3/03/25/5e0ba78bc27a3a089f137c9f1d92fdfce16d06996c071a016c
Successfully built openai-whisper
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https

In [ ]:
!pip install -q langchain-openai
!pip install -q langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.2 MB/s eta 0:00:00


In [ ]:
!pip -q install playsound

  Preparing metadata (setup.py) ... done


##Importing Libraries

In [ ]:
from IPython.display import Audio
import whisper
import os
import openai
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

##Play the audio file

In [ ]:
video = "PMDD_Consult.mp3" #Audio created using a text file from ChatGPT and then converting it into a Audio file using AI
Audio(video)

##Transcribe the audio file with the Whisper model

In [ ]:
model = whisper.load_model("base")
result = model.transcribe(video)
transcript = result["text"]
print("Transcription:", transcript)

100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 117MiB/s]


Transcription:  Hello, good afternoon. This is Dr. Harry speaking. Am I speaking with Ms. Hermione? Yes, hi, Dr. This is Hermione. Hi, Hermione. Thank you for joining the teleconcultation today. I understand you scheduled this appointment because you've been experiencing mood changes related to your menstrual cycle. Is that correct? Yes, that's right. For the past few months, I've noticed that about a week before my period, my mood becomes really difficult to manage. I see. Let's talk a little more about that. How old are you? And when was the first day of your last menstrual period? I am 24 years old and it started on the 4th of this month. Thank you. And are your cycles usually regular? Yes, mostly. My cycle is usually around 28 to 30 days. All right. Now you mentioned mood changes before your period. About how many days before your period do you start noticing these symptoms? Usually about 7 to 10 days before my period starts. And what kinds of mood changes do you experience during 

##Create Transcription file

In [ ]:
with open("transcript.txt", "w", encoding="utf-8") as txt:
    txt.write(transcript)
    print("File is ready")

File is ready


##Initialize the Large Language Model

In [ ]:
os.environ["OPENAI_API_KEY"] = ""
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.1, max_tokens= 512)

##Create a prompt template to Generate Summary

In [ ]:
#Prompt Template to use the application as a clinical teleconsultation summarizer
#Prompt drafted to avoid hallucinations
prompt_template ="""
You summarize clinical telehealth conversations related to women's reproductive mental and physical health.
Conversation: {conversation}
Rules to follow:
1. If there is no information found on what is asked, return that the information is missing. Do not make up an answer.
2. Strictly extract answer only from the transcript.
From the coversation, you will extract the following information:
1. Date of consultation
2. Patient's information - Name and Age
3. Chief Complaint and History of Present Illness
4. Menstrual History
5. Mental complaints / symptoms
6. Physical and associated complaints
7. List the medications if patient is currently taking any

Give a list of three differential diagnosis along with the reasons. Then mention the probable diagnoses while ruling out the other differential diagnoses.
Also list health check-up tests that are relevant to the probable diagnosis.
Provide a list of lifestyle modifications that can help the patient.
Then provide a check-list of data that would be required during the next follow-up visit.
"""

##Create the prompt and the chain

In [ ]:
clinical_prompt = PromptTemplate(
    input_variables=["conversation"],
    template=prompt_template,
)
clinical_chain = clinical_prompt | llm

##Execute the chain

In [ ]:
conversation = transcript
response = clinical_chain.invoke({"conversation": conversation})
print(response.content)


1. Date of consultation: Not provided
2. Patient's information: 
   - Name: Hermione
   - Age: 24 years old
3. Chief Complaint and History of Present Illness: 
   - Chief Complaint: Mood changes related to menstrual cycle
   - History of Present Illness: Experiencing irritability, emotional changes, anxiety, and feeling overwhelmed 7-10 days before menstrual period
4. Menstrual History: Regular cycles of 28-30 days, with symptoms occurring consistently for the past three months
5. Mental complaints/symptoms: Irritability, emotional changes, anxiety, feeling overwhelmed, sadness, hopelessness, loss of interest in activities
6. Physical and associated complaints: Bloating, breast tenderness, headaches, fatigue
7. Medications: Not provided

Differential Diagnosis:
1. Premenstrual Dysphoric Disorder (PMDD) - Symptoms of irritability, mood changes, anxiety, and physical symptoms occurring before menstruation.
2. Premenstrual Syndrome (PMS) - Similar symptoms to PMDD but less severe.
3. Majo